# Minería de Datos
## Taller: Asociación
## Datos del estudiante
- **Nombre:** Angel David Pineros Sierra
- **Asignatura:** Mineria de Datos
- **Tema:** Reglas de asociacion sobre market basket
- **Fecha:** 19 de abril de 2026

**Profesora:** Elizabeth León Guzmán


In [160]:
%pip install -q pyfpgrowth efficient-apriori


In [161]:
import numpy as np
import pandas as pd
import pyfpgrowth
import time
from efficient_apriori import apriori
from sklearn import preprocessing
from scipy import stats


## 1. Market Basket

Por cada un de las siguientes preguntas, proveer un ejemplo de una regla de asociación del dominio de “market basket” que
satisface las siguientes condiciones. También, describir si las reglas son interesantes (subjetivamente).

- **(a)** Una regla que que tiene alto soporte y alta confianza
- **(b)** Una regla que tiene razonablemente alto soporte pero baja confianza
- **(c)** Una regla que tiene bajo soporte y baja confianza
- **(d)** Una regla que tiene bajo soporte y alta confianza


### Carga del conjunto de datos

Se carga el archivo `marketBasket.csv`, se renombran las columnas para trabajar con nombres simples
y se convierten los valores `true/false` a variables booleanas.


In [162]:
basket_raw = pd.DataFrame(
    [
        [1, True, True, True, False, False, False],
        [2, True, False, False, True, True, False],
        [3, True, False, True, False, False, True],
        [4, False, False, False, True, True, True],
        [5, False, True, True, False, False, True],
        [6, True, False, True, True, True, False],
        [7, False, False, True, True, True, False],
        [8, False, True, True, False, False, False],
        [9, True, False, True, True, True, False],
        [10, False, True, False, False, False, True],
    ],
    columns=[
        "id",
        "leche",
        "cerveza",
        "panales",
        "pan",
        "mantequilla",
        "galletas",
    ],
)

basket = basket_raw.copy()
item_columns = [column for column in basket.columns if column != "id"]

basket


,id,leche,cerveza,panales,pan,mantequilla,galletas
0,1,True,True,True,False,False,False
1,2,True,False,False,True,True,False
2,3,True,False,True,False,False,True
3,4,False,False,False,True,True,True
4,5,False,True,True,False,False,True
5,6,True,False,True,True,True,False
6,7,False,False,True,True,True,False
7,8,False,True,True,False,False,False
8,9,True,False,True,True,True,False
9,10,False,True,False,False,False,True


### Exploración inicial

Para este ejercicio el soporte se interpreta como:

`soporte(itemset) = numero de transacciones que contienen el itemset / total de transacciones`

Como el dataset tiene 10 transacciones, un soporte de `0.50` equivale a `5` apariciones.


In [163]:
total_transactions = len(basket)

item_frequency = basket[item_columns].sum().astype(int)
item_support = item_frequency / total_transactions

item_summary = pd.DataFrame(
    {
        "item": item_columns,
        "frecuencia": item_frequency.values,
        "soporte": np.round(item_support.values, 2),
    }
).sort_values(["frecuencia", "item"], ascending=[False, True]).reset_index(drop=True)

item_summary


,item,frecuencia,soporte
0,panales,7,0.7
1,leche,5,0.5
2,mantequilla,5,0.5
3,pan,5,0.5
4,cerveza,4,0.4
5,galletas,4,0.4


### Preparación de transacciones

La tabla binaria se transforma en una lista de transacciones para poder evaluar las reglas de asociación
de forma directa y verificar los cálculos manualmente.


In [164]:
transactions = basket[item_columns].apply(
    lambda row: [item for item, present in row.items() if present],
    axis=1,
).tolist()

transaction_lengths = np.array([len(transaction) for transaction in transactions])
length_stats = stats.describe(transaction_lengths)

transactions_df = pd.DataFrame(
    {
        "tid": basket["id"].astype(int),
        "transaccion": [", ".join(transaction) for transaction in transactions],
        "numero_items": transaction_lengths,
    }
)

print(
    f"Cantidad de transacciones: {length_stats.nobs} | "
    f"Minimo de items: {length_stats.minmax[0]} | "
    f"Maximo de items: {length_stats.minmax[1]} | "
    f"Promedio de items: {length_stats.mean:.2f}"
)

transactions_df


Cantidad de transacciones: 10 | Minimo de items: 2 | Maximo de items: 4 | Promedio de items: 3.00


,tid,transaccion,numero_items
0,1,"leche, cerveza, panales",3
1,2,"leche, pan, mantequilla",3
2,3,"leche, panales, galletas",3
3,4,"pan, mantequilla, galletas",3
4,5,"cerveza, panales, galletas",3
5,6,"leche, panales, pan, mantequilla",4
6,7,"panales, pan, mantequilla",3
7,8,"cerveza, panales",2
8,9,"leche, panales, pan, mantequilla",4
9,10,"cerveza, galletas",2


### Reglas simples candidatas

Primero se revisan todas las reglas simples del tipo `A -> B`. Esto permite elegir ejemplos con evidencia
en la base y detectar si alguna categoría necesita una regla con antecedente de varios items.


In [165]:
def support_count(itemset):
    itemset = set(itemset)
    return sum(itemset.issubset(set(transaction)) for transaction in transactions)


def support(itemset):
    return support_count(itemset) / total_transactions


def confidence(lhs, rhs):
    lhs_count = support_count(lhs)
    return support_count(tuple(lhs) + tuple(rhs)) / lhs_count if lhs_count else np.nan


simple_rules_rows = []

for lhs in item_columns:
    for rhs in item_columns:
        if lhs == rhs:
            continue

        simple_rules_rows.append(
            {
                "regla": f"{lhs} -> {rhs}",
                "support_count": support_count((lhs, rhs)),
                "support": round(support((lhs, rhs)), 2),
                "confidence": round(confidence((lhs,), (rhs,)), 2),
            }
        )

simple_rules_df = pd.DataFrame(simple_rules_rows).sort_values(
    ["support", "confidence", "regla"],
    ascending=[False, False, True],
).reset_index(drop=True)

simple_rules_df


,regla,support_count,support,confidence
0,mantequilla -> pan,5,0.5,1.00
1,pan -> mantequilla,5,0.5,1.00
2,leche -> panales,4,0.4,0.80
3,panales -> leche,4,0.4,0.57
4,cerveza -> panales,3,0.3,0.75
5,leche -> mantequilla,3,0.3,0.60
6,leche -> pan,3,0.3,0.60
7,mantequilla -> leche,3,0.3,0.60
8,mantequilla -> panales,3,0.3,0.60
9,pan -> leche,3,0.3,0.60


### Desarrollo

Se seleccionan cuatro ejemplos concretos para responder los incisos `(a)` a `(d)`. En el inciso `(d)`
se usa un antecedente multiple porque en las reglas simples `1 -> 1` no aparece una combinacion que tenga
simultaneamente bajo soporte y alta confianza.


In [166]:
selected_rules = [
    {
        "inciso": "(a)",
        "categoria": "Alto soporte y alta confianza",
        "lhs": ("pan",),
        "rhs": ("mantequilla",),
    },
    {
        "inciso": "(b)",
        "categoria": "Soporte razonablemente alto y baja confianza",
        "lhs": ("panales",),
        "rhs": ("leche",),
    },
    {
        "inciso": "(c)",
        "categoria": "Bajo soporte y baja confianza",
        "lhs": ("cerveza",),
        "rhs": ("leche",),
    },
    {
        "inciso": "(d)",
        "categoria": "Bajo soporte y alta confianza",
        "lhs": ("leche", "cerveza"),
        "rhs": ("panales",),
    },
]

summary_rows = []

for rule in selected_rules:
    union = tuple(rule["lhs"]) + tuple(rule["rhs"])
    union_count = support_count(union)
    lhs_count = support_count(rule["lhs"])

    summary_rows.append(
        {
            "inciso": rule["inciso"],
            "categoria": rule["categoria"],
            "regla": f"{', '.join(rule['lhs'])} -> {', '.join(rule['rhs'])}",
            "support_count": union_count,
            "support": round(union_count / total_transactions, 2),
            "confidence": round(union_count / lhs_count, 2),
        }
    )

selected_rules_df = pd.DataFrame(summary_rows)
selected_rules_df


,inciso,categoria,regla,support_count,support,confidence
0,(a),Alto soporte y alta confianza,pan -> mantequilla,5,0.5,1.00
1,(b),Soporte razonablemente alto y baja confianza,panales -> leche,4,0.4,0.57
2,(c),Bajo soporte y baja confianza,cerveza -> leche,1,0.1,0.25
3,(d),Bajo soporte y alta confianza,"leche, cerveza -> panales",1,0.1,1.00


### Justificación de cada regla

**(a) `pan -> mantequilla`**

$$
\operatorname{support}(pan \rightarrow mantequilla) = \frac{5}{10} = 0.50
$$

$$
\operatorname{conf}(pan \rightarrow mantequilla) = \frac{5}{5} = 1.00
$$

- **Interés subjetivo:** Sí.
- **Justificación:** combina una frecuencia alta con cumplimiento total, por lo que representa una asociación fuerte y estable en esta base.

**(b) `panales -> leche`**

$$
\operatorname{support}(panales \rightarrow leche) = \frac{4}{10} = 0.40
$$

$$
\operatorname{conf}(panales \rightarrow leche) = \frac{4}{7} \approx 0.57
$$

- **Interés subjetivo:** Parcial.
- **Justificación:** es una regla relativamente frecuente, pero la confianza no es lo bastante alta como para considerarla una relación fuerte.

**(c) `cerveza -> leche`**

$$
\operatorname{support}(cerveza \rightarrow leche) = \frac{1}{10} = 0.10
$$

$$
\operatorname{conf}(cerveza \rightarrow leche) = \frac{1}{4} = 0.25
$$

- **Interés subjetivo:** No.
- **Justificación:** ocurre muy pocas veces y además falla en la mayoría de transacciones donde aparece cerveza.

**(d) `leche, cerveza -> panales`**

$$
\operatorname{support}(leche, cerveza \rightarrow panales) = \frac{1}{10} = 0.10
$$

$$
\operatorname{conf}(leche, cerveza \rightarrow panales) = \frac{1}{1} = 1.00
$$

- **Interés subjetivo:** Sí, pero limitada.
- **Justificación:** la confianza es perfecta, pero la regla solo se observa una vez; sirve como patrón puntual, no como tendencia general.


## 2. Descubrimiento de reglas de asociación

¿Por qué el proceso de descubrimiento de reglas de asociación es relativamente simple comparado con la generación de grandes conjuntos de ítems en bases de datos transaccionales?

Porque la parte realmente costosa no suele ser formular las reglas, sino encontrar primero los conjuntos de ítems frecuentes dentro de una base transaccional grande.

Una vez que ya se tienen esos conjuntos frecuentes, generar reglas es relativamente directo:
- se toma un subconjunto como antecedente,
- el resto queda como consecuente,
- y la confianza se calcula con soportes que ya fueron obtenidos.

En cambio, la generación de grandes itemsets es más compleja porque:
- hay una explosión combinatoria de candidatos,
- se deben evaluar muchísimas combinaciones de ítems,
- normalmente se requieren varias pasadas sobre la base de datos,
- y muchas combinaciones terminan no siendo frecuentes.


## 3. Conjunto de datos supermercado

Considere el siguiente conjunto de datos:

| TID | Ítems |
|---|---|
| T01 | milk, beer, diapers |
| T02 | bread, butter, milk |
| T03 | milk, diapers, cookies |
| T04 | bread, butter, cookies |
| T05 | beer, cookies, diapers |
| T06 | milk, diapers, bread, butter |
| T07 | bread, butter, diapers |
| T08 | beer, diapers |
| T09 | milk, diapers, bread, butter |
| T10 | beer, cookies |

Resolver:
- **(a)** ¿cuál es el número máximo de reglas de asociación que se pueden generar? (incluyendo reglas con soporte 0)
- **(b)** ¿cuál es el tamaño máximo de los conjuntos de items frecuentes que se pueden extraer (asumir `minsoporte > 0`)?
- **(c)** Escribir una regla que contenga 3 ítems que se genere de este conjunto de datos.
- **(d)** Encontrar un conjunto de items (de tamaño mayor a 2) con el valor de soporte máximo.
- **(e)** Encontrar un par de ítems `(a, b)` tal que las reglas `a -> b` y `b -> a` tengan la misma confianza.


In [167]:
from itertools import combinations
from math import comb

point3_transactions = [
    ("milk", "beer", "diapers"),
    ("bread", "butter", "milk"),
    ("milk", "diapers", "cookies"),
    ("bread", "butter", "cookies"),
    ("beer", "cookies", "diapers"),
    ("milk", "diapers", "bread", "butter"),
    ("bread", "butter", "diapers"),
    ("beer", "diapers"),
    ("milk", "diapers", "bread", "butter"),
    ("beer", "cookies"),
]

point3_df = pd.DataFrame(
    {
        "TID": [f"T{i:02d}" for i in range(1, len(point3_transactions) + 1)],
        "Items": [", ".join(transaction) for transaction in point3_transactions],
        "Numero_items": [len(transaction) for transaction in point3_transactions],
    }
)

point3_items = sorted({item for transaction in point3_transactions for item in transaction})
point3_n = len(point3_transactions)
point3_df


,TID,Items,Numero_items
0,T01,"milk, beer, diapers",3
1,T02,"bread, butter, milk",3
2,T03,"milk, diapers, cookies",3
3,T04,"bread, butter, cookies",3
4,T05,"beer, cookies, diapers",3
5,T06,"milk, diapers, bread, butter",4
6,T07,"bread, butter, diapers",3
7,T08,"beer, diapers",2
8,T09,"milk, diapers, bread, butter",4
9,T10,"beer, cookies",2


In [168]:
def point3_support_count(itemset):
    itemset = set(itemset)
    return sum(itemset.issubset(set(transaction)) for transaction in point3_transactions)


def point3_support(itemset):
    return point3_support_count(itemset) / point3_n


def point3_confidence(lhs, rhs):
    lhs_count = point3_support_count(lhs)
    return point3_support_count(tuple(lhs) + tuple(rhs)) / lhs_count if lhs_count else np.nan


point3_max_rules = 3 ** len(point3_items) - 2 ** (len(point3_items) + 1) + 1
point3_max_rule_check = sum(comb(len(point3_items), k) * ((2 ** k) - 2) for k in range(1, len(point3_items) + 1))
point3_max_itemset_size = max(len(transaction) for transaction in point3_transactions)

point3_large_itemsets = []
for size in range(3, len(point3_items) + 1):
    for itemset in combinations(point3_items, size):
        count = point3_support_count(itemset)
        if count > 0:
            point3_large_itemsets.append(
                {
                    "itemset": ", ".join(itemset),
                    "size": size,
                    "support_count": count,
                    "support": round(count / point3_n, 2),
                }
            )

point3_large_itemsets_df = pd.DataFrame(point3_large_itemsets).sort_values(
    ["support_count", "size", "itemset"],
    ascending=[False, False, True],
).reset_index(drop=True)

point3_equal_confidence_pairs = []
for left_item, right_item in combinations(point3_items, 2):
    left_confidence = point3_confidence((left_item,), (right_item,))
    right_confidence = point3_confidence((right_item,), (left_item,))
    if np.isclose(left_confidence, right_confidence):
        point3_equal_confidence_pairs.append(
            {
                "a": left_item,
                "b": right_item,
                "conf(a->b)": round(left_confidence, 2),
                "conf(b->a)": round(right_confidence, 2),
            }
        )

point3_summary = pd.DataFrame(
    [
        {"resultado": "Número de ítems distintos", "valor": len(point3_items)},
        {"resultado": "Máximo de reglas posibles", "valor": point3_max_rules},
        {"resultado": "Chequeo por sumatoria", "valor": point3_max_rule_check},
        {"resultado": "Tamaño máximo de itemset frecuente", "valor": point3_max_itemset_size},
    ]
)

print("Resumen base")
display(point3_summary)

print("Itemsets de tamaño mayor a 2 ordenados por soporte")
display(point3_large_itemsets_df)

print("Pares de ítems con la misma confianza en ambos sentidos")
display(pd.DataFrame(point3_equal_confidence_pairs))


Resumen base


,resultado,valor
0,Número de ítems distintos,6
1,Máximo de reglas posibles,602
2,Chequeo por sumatoria,602
3,Tamaño máximo de itemset frecuente,4


Itemsets de tamaño mayor a 2 ordenados por soporte


,itemset,size,support_count,support
0,"bread, butter, diapers",3,3,0.3
1,"bread, butter, milk",3,3,0.3
2,"bread, butter, diapers, milk",4,2,0.2
3,"bread, diapers, milk",3,2,0.2
4,"butter, diapers, milk",3,2,0.2
5,"beer, cookies, diapers",3,1,0.1
6,"beer, diapers, milk",3,1,0.1
7,"bread, butter, cookies",3,1,0.1
8,"cookies, diapers, milk",3,1,0.1


Pares de ítems con la misma confianza en ambos sentidos


,a,b,conf(a->b),conf(b->a)
0,beer,bread,0.0,0.0
1,beer,butter,0.0,0.0
2,beer,cookies,0.5,0.5
3,bread,butter,1.0,1.0
4,bread,milk,0.6,0.6
5,butter,milk,0.6,0.6


**(a) Número máximo de reglas de asociación**
- Hay $6$ ítems distintos: `beer`, `bread`, `butter`, `cookies`, `diapers`, `milk`.
- El número máximo de reglas posibles, incluyendo reglas con soporte $0$, es:

$$
3^6 - 2^{6+1} + 1 = 729 - 128 + 1 = 602
$$

- **Respuesta:** $602$ reglas.

**(b) Tamaño máximo de los conjuntos de ítems frecuentes con `minsoporte > 0`**
- Basta con buscar el tamaño máximo de un itemset que aparezca al menos una vez.
- En la base, las transacciones `T06` y `T09` tienen $4$ ítems.

$$
\max |X| = 4
$$

**(c) Una regla que contenga $3$ ítems**
- Un ejemplo válido es:

$$
bread, butter \rightarrow diapers
$$

- Esta regla se genera a partir del itemset $\{bread, butter, diapers\}$.

**(d) Conjunto de ítems de tamaño mayor a $2$ con soporte máximo**
- Hay empate en el soporte máximo entre dos itemsets:

$$
\operatorname{support}(\{bread, butter, diapers\}) = \frac{3}{10} = 0.30
$$

$$
\operatorname{support}(\{bread, butter, milk\}) = \frac{3}{10} = 0.30
$$

- **Respuesta:** cualquiera de los dos es correcto; ambos tienen el soporte máximo.

**(e) Par de ítems $(a, b)$ tal que $a \rightarrow b$ y $b \rightarrow a$ tengan la misma confianza**
- Un ejemplo es `bread` y `butter`.

$$
\operatorname{conf}(bread \rightarrow butter) = \frac{5}{5} = 1.00
$$

$$
\operatorname{conf}(butter \rightarrow bread) = \frac{5}{5} = 1.00
$$

- **Respuesta:** el par `bread, butter` cumple la condición.


## 4. Base de datos X

Dada la siguiente base de datos X:

| TID | Ítems |
|---|---|
| T01 | A, B, C, D |
| T02 | A, C, D, F |
| T03 | C, D, E, G, A |
| T04 | A, D, F, B |
| T05 | B, C, G |
| T06 | D, F, G |
| T07 | A, B, G |
| T08 | C, D, F, G |

Usando valores de umbral de soporte = 25 % y confianza = 60 %, encuentre:

- **(a)** Todos los conjuntos de ítems en la base de datos X
- **(b)** Reglas de asociación fuertes para la base de datos X
- **(c)** Analice las asociaciones engañosas para el conjunto de reglas obtenido en el numeral anterior.


In [169]:
from itertools import combinations

point4_transactions = {
    "T01": {"A", "B", "C", "D"},
    "T02": {"A", "C", "D", "F"},
    "T03": {"C", "D", "E", "G", "A"},
    "T04": {"A", "D", "F", "B"},
    "T05": {"B", "C", "G"},
    "T06": {"D", "F", "G"},
    "T07": {"A", "B", "G"},
    "T08": {"C", "D", "F", "G"},
}

point4_df = pd.DataFrame(
    {
        "TID": list(point4_transactions.keys()),
        "Items": [", ".join(sorted(items)) for items in point4_transactions.values()],
        "Numero_items": [len(items) for items in point4_transactions.values()],
    }
)

point4_items = sorted(set().union(*point4_transactions.values()))
point4_n = len(point4_transactions)
point4_min_support = 0.25
point4_min_support_count = int(point4_n * point4_min_support)
point4_min_confidence = 0.60

point4_df


,TID,Items,Numero_items
0,T01,"A, B, C, D",4
1,T02,"A, C, D, F",4
2,T03,"A, C, D, E, G",5
3,T04,"A, B, D, F",4
4,T05,"B, C, G",3
5,T06,"D, F, G",3
6,T07,"A, B, G",3
7,T08,"C, D, F, G",4


In [170]:
def point4_support_count(itemset):
    itemset = set(itemset)
    return sum(itemset.issubset(transaction) for transaction in point4_transactions.values())


def point4_support(itemset):
    return point4_support_count(itemset) / point4_n


def point4_confidence(lhs, rhs):
    lhs_count = point4_support_count(lhs)
    return point4_support_count(tuple(lhs) + tuple(rhs)) / lhs_count if lhs_count else np.nan


point4_itemsets = []
for size in range(1, len(point4_items) + 1):
    for itemset in combinations(point4_items, size):
        count = point4_support_count(itemset)
        if count >= point4_min_support_count:
            point4_itemsets.append(
                {
                    "itemset": "{" + ", ".join(itemset) + "}",
                    "size": size,
                    "support_count": count,
                    "support": round(count / point4_n, 3),
                }
            )

point4_itemsets_df = pd.DataFrame(point4_itemsets).sort_values(
    ["size", "itemset"],
    ascending=[True, True],
).reset_index(drop=True)

point4_support_map = {
    tuple(sorted(row["itemset"].strip("{}").split(", "))) if row["itemset"] != "{}" else tuple(): row["support_count"]
    for _, row in point4_itemsets_df.iterrows()
}

point4_rules = []
for _, row in point4_itemsets_df.iterrows():
    itemset = tuple(row["itemset"].strip("{}").split(", "))
    if len(itemset) < 2:
        continue

    support_count = row["support_count"]
    support_value = support_count / point4_n
    itemset_set = set(itemset)

    for lhs_size in range(1, len(itemset)):
        for lhs in combinations(itemset, lhs_size):
            rhs = tuple(sorted(itemset_set - set(lhs)))
            lhs_support_count = point4_support_count(lhs)
            rhs_support_count = point4_support_count(rhs)
            confidence_value = support_count / lhs_support_count

            if confidence_value >= point4_min_confidence:
                lift_value = support_value / ((lhs_support_count / point4_n) * (rhs_support_count / point4_n))
                point4_rules.append(
                    {
                        "antecedente": "{" + ", ".join(lhs) + "}",
                        "consecuente": "{" + ", ".join(rhs) + "}",
                        "support": round(support_value, 3),
                        "confidence": round(confidence_value, 3),
                        "lift": round(lift_value, 3),
                    }
                )

point4_rules_df = pd.DataFrame(point4_rules).sort_values(
    ["confidence", "lift", "antecedente", "consecuente"],
    ascending=[False, False, True, True],
).reset_index(drop=True)

point4_misleading_df = point4_rules_df[point4_rules_df["lift"] <= 1].reset_index(drop=True)

point4_summary_df = pd.DataFrame(
    [
        {"medida": "Número de ítems distintos", "valor": len(point4_items)},
        {"medida": "Soporte mínimo", "valor": point4_min_support},
        {"medida": "Soporte mínimo absoluto", "valor": point4_min_support_count},
        {"medida": "Confianza mínima", "valor": point4_min_confidence},
        {"medida": "Cantidad de itemsets frecuentes", "valor": len(point4_itemsets_df)},
        {"medida": "Cantidad de reglas fuertes", "valor": len(point4_rules_df)},
    ]
)

print("Resumen del punto 4")
display(point4_summary_df)

print("Conjuntos de ítems frecuentes")
display(point4_itemsets_df)

print("Reglas de asociación fuertes")
display(point4_rules_df)

print("Reglas fuertes potencialmente engañosas (lift <= 1)")
display(point4_misleading_df)


Resumen del punto 4


,medida,valor
0,Número de ítems distintos,7.00
1,Soporte mínimo,0.25
2,Soporte mínimo absoluto,2.00
3,Confianza mínima,0.60
4,Cantidad de itemsets frecuentes,26.00
5,Cantidad de reglas fuertes,26.00


Conjuntos de ítems frecuentes


,itemset,size,support_count,support
0,{A},1,5,0.625
1,{B},1,4,0.500
2,{C},1,5,0.625
3,{D},1,6,0.750
4,{F},1,4,0.500
5,{G},1,5,0.625
6,"{A, B}",2,3,0.375
7,"{A, C}",2,3,0.375
8,"{A, D}",2,4,0.500
9,"{A, F}",2,2,0.250


Reglas de asociación fuertes


,antecedente,consecuente,support,confidence,lift
0,"{B, D}",{A},0.250,1.000,1.600
1,"{A, C}",{D},0.375,1.000,1.333
2,"{A, F}",{D},0.250,1.000,1.333
3,"{C, F}",{D},0.250,1.000,1.333
4,"{F, G}",{D},0.250,1.000,1.333
5,{F},{D},0.500,1.000,1.333
6,{A},{D},0.500,0.800,1.067
7,{C},{D},0.500,0.800,1.067
8,"{A, D}",{C},0.375,0.750,1.200
9,{B},{A},0.375,0.750,1.200


Reglas fuertes potencialmente engañosas (lift <= 1)


,antecedente,consecuente,support,confidence,lift
0,"{A, B}",{D},0.250,0.667,0.889
1,"{C, G}",{D},0.250,0.667,0.889
2,{A},{C},0.375,0.600,0.960
3,{C},{A},0.375,0.600,0.960
4,{C},{G},0.375,0.600,0.960
5,{G},{C},0.375,0.600,0.960
6,{G},{D},0.375,0.600,0.800



**(a) Todos los conjuntos de ítems frecuentes en la base $X$**
- Con soporte mínimo del $25\%$, el soporte mínimo absoluto es:

$$
\text{minsupport} = \frac{2}{8} = 0.25
$$

- Los itemsets frecuentes son:
  - Tamaño $1$: $\{A\}, \{B\}, \{C\}, \{D\}, \{F\}, \{G\}$
  - Tamaño $2$: $\{A,B\}, \{A,C\}, \{A,D\}, \{A,F\}, \{A,G\}, \{B,C\}, \{B,D\}, \{B,G\}, \{C,D\}, \{C,F\}, \{C,G\}, \{D,F\}, \{D,G\}, \{F,G\}$
  - Tamaño $3$: $\{A,B,D\}, \{A,C,D\}, \{A,D,F\}, \{C,D,F\}, \{C,D,G\}, \{D,F,G\}$
- En total hay $26$ conjuntos de ítems frecuentes.

**(b) Reglas de asociación fuertes para la base $X$**
- Son fuertes las reglas que cumplen:

$$
\operatorname{support} \ge 0.25 \quad \text{y} \quad \operatorname{confidence} \ge 0.60
$$

- La tabla de la celda anterior contiene el conjunto completo de reglas fuertes.
- Algunas de las más representativas son:

$$
\{B\} \rightarrow \{A\} \quad \operatorname{conf} = 0.75
$$

$$
\{A\} \rightarrow \{D\} \quad \operatorname{conf} = 0.80
$$

$$
\{C\} \rightarrow \{D\} \quad \operatorname{conf} = 0.80
$$

$$
\{F\} \rightarrow \{D\} \quad \operatorname{conf} = 1.00
$$

$$
\{A,C\} \rightarrow \{D\} \quad \operatorname{conf} = 1.00
$$

$$
\{B,D\} \rightarrow \{A\} \quad \operatorname{conf} = 1.00
$$

$$
\{F,G\} \rightarrow \{D\} \quad \operatorname{conf} = 1.00
$$

**(c) Asociaciones engañosas**
- Para este análisis, se consideran engañosas las reglas fuertes con:

$$
\operatorname{lift} \le 1
$$

- Estas reglas cumplen soporte y confianza, pero no muestran una asociación positiva real; en algunos casos el consecuente ya es muy frecuente por sí mismo.
- Ejemplos claros de reglas engañosas en esta base son:

$$
\{A\} \rightarrow \{C\} \quad \operatorname{lift} = 0.96
$$

$$
\{C\} \rightarrow \{A\} \quad \operatorname{lift} = 0.96
$$

$$
\{C\} \rightarrow \{G\} \quad \operatorname{lift} = 0.96
$$

$$
\{G\} \rightarrow \{C\} \quad \operatorname{lift} = 0.96
$$

$$
\{G\} \rightarrow \{D\} \quad \operatorname{lift} = 0.80
$$

$$
\{A,B\} \rightarrow \{D\} \quad \operatorname{lift} = 0.889
$$

$$
\{C,G\} \rightarrow \{D\} \quad \operatorname{lift} = 0.889
$$

- La conclusión es que una regla puede ser fuerte por confianza y, aun así, resultar engañosa si el valor de $\operatorname{lift}$ no supera $1$.


## 5. Apriori y lattice de candidatos

El algoritmo Apriori usa estrategias de generación y conteo para derivar conjuntos de ítems frecuentes. Conjuntos de ítem de tamaño $k + 1$ son creados de conjuntos de ítems de tamaño $k$. Un conjunto candidato es eliminado si uno de sus subconjuntos no es frecuente en la fase de poda. Supongamos que el algoritmo Apriori es aplicado a los datos de la siguiente tabla con un soporte mínimo de $30\%$.

| id | items |
|---|---|
| 1 | a,b,d,e |
| 2 | b,c,d |
| 3 | a,b,d,e |
| 4 | a,c,d,e |
| 5 | b,c,d,e |
| 6 | b,d,e |
| 7 | c,d |
| 8 | a,b,c |
| 9 | a,d,e |
| 10 | b,d |

Resolver:
- **(a)** Dibujar el lattice de los conjuntos de items y etiquetar cada nodo con `N`, `F` o `I`.
- **(b)** ¿Cuál es el porcentaje de conjuntos de items frecuente?
- **(c)** ¿Cuál es el radio de poda en este conjunto de datos?
- **(d)** ¿Cuál es la rata de falsa alarma?


In [171]:
from itertools import combinations

point5_transactions = [
    {"a", "b", "d", "e"},
    {"b", "c", "d"},
    {"a", "b", "d", "e"},
    {"a", "c", "d", "e"},
    {"b", "c", "d", "e"},
    {"b", "d", "e"},
    {"c", "d"},
    {"a", "b", "c"},
    {"a", "d", "e"},
    {"b", "d"},
]

point5_items = sorted(set().union(*point5_transactions))
point5_n = len(point5_transactions)
point5_min_support = 0.30
point5_min_support_count = 3


def point5_support_count(itemset):
    itemset = set(itemset)
    return sum(itemset.issubset(transaction) for transaction in point5_transactions)


def point5_generate_candidates(previous_frequent, level):
    previous_frequent = sorted(previous_frequent)
    previous_set = set(previous_frequent)
    generated = set()

    for left_index in range(len(previous_frequent)):
        for right_index in range(left_index + 1, len(previous_frequent)):
            left_itemset = previous_frequent[left_index]
            right_itemset = previous_frequent[right_index]
            union = tuple(sorted(set(left_itemset) | set(right_itemset)))

            if len(union) != level:
                continue

            if all(tuple(sorted(subset)) in previous_set for subset in combinations(union, level - 1)):
                generated.add(union)

    return sorted(generated)


point5_candidate_levels = {}
point5_frequent_levels = {}

point5_candidate_levels[1] = [(item,) for item in point5_items]
point5_frequent_levels[1] = [
    itemset for itemset in point5_candidate_levels[1]
    if point5_support_count(itemset) >= point5_min_support_count
]

level = 2
while point5_frequent_levels.get(level - 1):
    point5_candidate_levels[level] = point5_generate_candidates(point5_frequent_levels[level - 1], level)
    point5_frequent_levels[level] = [
        itemset for itemset in point5_candidate_levels[level]
        if point5_support_count(itemset) >= point5_min_support_count
    ]

    if not point5_candidate_levels[level]:
        break

    level += 1

point5_all_subsets = [
    subset for size in range(1, len(point5_items) + 1)
    for subset in combinations(point5_items, size)
]
point5_candidate_set = {
    subset for subsets in point5_candidate_levels.values() for subset in subsets
}
point5_frequent_set = {
    subset for subsets in point5_frequent_levels.values() for subset in subsets
}
point5_infrequent_candidate_set = point5_candidate_set - point5_frequent_set
point5_not_candidate_set = set(point5_all_subsets) - point5_candidate_set

point5_level_rows = []
for level_key in sorted(point5_candidate_levels.keys()):
    candidates = point5_candidate_levels[level_key]
    frequent = point5_frequent_levels[level_key]
    infrequent = sorted(set(candidates) - set(frequent))
    point5_level_rows.append(
        {
            "nivel": level_key,
            "Ck": ", ".join("{" + ",".join(itemset) + "}" for itemset in candidates) if candidates else "∅",
            "Fk": ", ".join("{" + ",".join(itemset) + "}" for itemset in frequent) if frequent else "∅",
            "Ik": ", ".join("{" + ",".join(itemset) + "}" for itemset in infrequent) if infrequent else "∅",
        }
    )

point5_levels_df = pd.DataFrame(point5_level_rows)

point5_lattice_rows = []
for subset in point5_all_subsets:
    if subset in point5_frequent_set:
        label = "F"
    elif subset in point5_infrequent_candidate_set:
        label = "I"
    else:
        label = "N"

    point5_lattice_rows.append(
        {
            "itemset": "{" + ",".join(subset) + "}",
            "size": len(subset),
            "support_count": point5_support_count(subset),
            "label": label,
        }
    )

point5_lattice_df = pd.DataFrame(point5_lattice_rows).sort_values(
    ["size", "itemset"],
    ascending=[True, True],
).reset_index(drop=True)

point5_frequent_percentage = len(point5_frequent_set) / len(point5_all_subsets) * 100
point5_pruning_ratio = len(point5_not_candidate_set) / len(point5_all_subsets) * 100
point5_false_alarm_rate = len(point5_infrequent_candidate_set) / len(point5_candidate_set) * 100

point5_summary_df = pd.DataFrame(
    [
        {"medida": "Subconjuntos no vacíos", "valor": len(point5_all_subsets)},
        {"medida": "Candidatos Apriori", "valor": len(point5_candidate_set)},
        {"medida": "Frecuentes", "valor": len(point5_frequent_set)},
        {"medida": "Candidatos infrecuentes", "valor": len(point5_infrequent_candidate_set)},
        {"medida": "No candidatos", "valor": len(point5_not_candidate_set)},
        {"medida": "% frecuentes", "valor": round(point5_frequent_percentage, 2)},
        {"medida": "% radio de poda", "valor": round(point5_pruning_ratio, 2)},
        {"medida": "% falsa alarma", "valor": round(point5_false_alarm_rate, 2)},
    ]
)

print("Resumen del punto 5")
display(point5_summary_df)

print("Niveles del algoritmo Apriori")
display(point5_levels_df)

print("Clasificación completa del lattice")
display(point5_lattice_df)


Resumen del punto 5


,medida,valor
0,Subconjuntos no vacíos,31.00
1,Candidatos Apriori,20.00
2,Frecuentes,15.00
3,Candidatos infrecuentes,5.00
4,No candidatos,11.00
5,% frecuentes,48.39
6,% radio de poda,35.48
7,% falsa alarma,25.00


Niveles del algoritmo Apriori


,nivel,Ck,Fk,Ik
0,1,"{a}, {b}, {c}, {d}, {e}","{a}, {b}, {c}, {d}, {e}",∅
1,2,"{a,b}, {a,c}, {a,d}, {a,e}, {b,c}, {b,d}, {b,e...","{a,b}, {a,d}, {a,e}, {b,c}, {b,d}, {b,e}, {c,d...","{a,c}, {c,e}"
2,3,"{a,b,d}, {a,b,e}, {a,d,e}, {b,c,d}, {b,d,e}","{a,d,e}, {b,d,e}","{a,b,d}, {a,b,e}, {b,c,d}"
3,4,∅,∅,∅


Clasificación completa del lattice


,itemset,size,support_count,label
0,{a},1,5,F
1,{b},1,7,F
2,{c},1,5,F
3,{d},1,9,F
4,{e},1,6,F
5,"{a,b}",2,3,F
6,"{a,c}",2,2,I
7,"{a,d}",2,4,F
8,"{a,e}",2,4,F
9,"{b,c}",2,3,F


### Lattice de candidatos


**(a) Lattice de candidatos con etiquetas**
- La clasificación completa de los $31$ subconjuntos no vacíos aparece en `point5_lattice_df`.
- Los nodos frecuentes (`F`) son:
  $\{a\}, \{b\}, \{c\}, \{d\}, \{e\}, \{a,b\}, \{a,d\}, \{a,e\}, \{b,c\}, \{b,d\}, \{b,e\}, \{c,d\}, \{d,e\}, \{a,d,e\}, \{b,d,e\}$.
- Los candidatos infrecuentes (`I`) son:
  $\{a,c\}, \{c,e\}, \{a,b,d\}, \{a,b,e\}, \{b,c,d\}$.
- Los no candidatos (`N`) son:
  $\{a,b,c\}, \{a,c,d\}, \{a,c,e\}, \{b,c,e\}, \{c,d,e\}, \{a,b,c,d\}, \{a,b,c,e\}, \{a,b,d,e\}, \{a,c,d,e\}, \{b,c,d,e\}, \{a,b,c,d,e\}$.

**(b) Porcentaje de conjuntos de ítems frecuentes**

$$
\frac{15}{31} \times 100 = 48.39\%
$$


**(c) Radio de poda**
- El radio de poda se calcula como el porcentaje de subconjuntos que no fueron considerados candidatos.

$$
\frac{11}{31} \times 100 = 35.48\%
$$


**(d) Tasa de falsa alarma**
- La tasa de falsa alarma es el porcentaje de candidatos que resultaron infrecuentes después de contar soporte.

$$
\frac{5}{20} \times 100 = 25.00\%
$$


## 6. Base de datos Y: Apriori vs FP-Growth

Dada la siguiente base de datos transaccional $Y$:

| TID | Ítems |
|---|---|
| 1 | A, B, C |
| 2 | A, C, D, E |
| 3 | A, B, D |
| 4 | A, C, F |
| 5 | A, B |
| 6 | A, E, F |
| 7 | A, B, D, E, F |
| 8 | A, F |
| 9 | B, D, E |
| 10 | B, D, E, F |
| 11 | B, C, D, E |
| 12 | C, D, E |

Usar Apriori y FP-Growth para encontrar los conjuntos de ítems frecuentes con mínimo soporte de `2`. Tratar con varios valores de confianza. Ordenar los conjuntos resultados y analizar resultados de cada algoritmo. Comparar eficiencia de los dos procesos de minería. Repetir con soporte de `3` y comparar los resultados.


In [172]:
from itertools import combinations

point6_transactions = [
    ("A", "B", "C"),
    ("A", "C", "D", "E"),
    ("A", "B", "D"),
    ("A", "C", "F"),
    ("A", "B"),
    ("A", "E", "F"),
    ("A", "B", "D", "E", "F"),
    ("A", "F"),
    ("B", "D", "E"),
    ("B", "D", "E", "F"),
    ("B", "C", "D", "E"),
    ("C", "D", "E"),
]

point6_transactions = [tuple(sorted(transaction)) for transaction in point6_transactions]
point6_n = len(point6_transactions)
point6_support_counts = [2, 3]
point6_confidences = [0.4, 0.5, 0.6, 0.8]
point6_timing_repetitions = 100

point6_dataset_df = pd.DataFrame(
    {
        "TID": [f"T{i:02d}" for i in range(1, point6_n + 1)],
        "items": [", ".join(transaction) for transaction in point6_transactions],
    }
)

point6_dataset_df


,TID,items
0,T01,"A, B, C"
1,T02,"A, C, D, E"
2,T03,"A, B, D"
3,T04,"A, C, F"
4,T05,"A, B"
5,T06,"A, E, F"
6,T07,"A, B, D, E, F"
7,T08,"A, F"
8,T09,"B, D, E"
9,T10,"B, D, E, F"


In [173]:
def point6_normalize_itemset(itemset):
    return tuple(sorted(itemset))


def point6_itemset_label(itemset):
    return "{" + ", ".join(itemset) + "}"


def point6_support_count(itemset):
    itemset = set(itemset)
    return sum(itemset.issubset(set(transaction)) for transaction in point6_transactions)


def point6_itemset_rows_from_lookup(support_lookup, algorithm, min_support_count):
    rows = []
    for itemset, support_count in sorted(support_lookup.items(), key=lambda pair: (len(pair[0]), pair[0])):
        rows.append(
            {
                "algoritmo": algorithm,
                "min_support_count": min_support_count,
                "itemset": point6_itemset_label(itemset),
                "size": len(itemset),
                "support_count": support_count,
                "support": round(support_count / point6_n, 4),
            }
        )
    return rows


def point6_generate_rules_from_lookup(support_lookup, algorithm, min_support_count, min_confidence):
    rows = []
    for itemset, support_count_union in sorted(support_lookup.items(), key=lambda pair: (len(pair[0]), pair[0])):
        if len(itemset) < 2:
            continue

        itemset_set = set(itemset)
        for lhs_size in range(1, len(itemset)):
            for lhs in combinations(itemset, lhs_size):
                lhs = point6_normalize_itemset(lhs)
                rhs = point6_normalize_itemset(itemset_set.difference(lhs))
                lhs_support_count = support_lookup.get(lhs, point6_support_count(lhs))
                confidence = round(support_count_union / lhs_support_count, 4)

                if confidence >= min_confidence:
                    rows.append(
                        {
                            "algoritmo": algorithm,
                            "min_support_count": min_support_count,
                            "min_confidence": min_confidence,
                            "antecedente": point6_itemset_label(lhs),
                            "consecuente": point6_itemset_label(rhs),
                            "antecedente_size": len(lhs),
                            "consecuente_size": len(rhs),
                            "support_count_union": support_count_union,
                            "support_union": round(support_count_union / point6_n, 4),
                            "confidence": confidence,
                        }
                    )

    return sorted(
        rows,
        key=lambda row: (
            row["antecedente_size"],
            row["antecedente"],
            row["consecuente"],
            -row["confidence"],
        ),
    )


def point6_average_runtime(callback, repetitions=point6_timing_repetitions):
    start = time.perf_counter()
    result = None
    for _ in range(repetitions):
        result = callback()
    elapsed_seconds = (time.perf_counter() - start) / repetitions
    return result, elapsed_seconds


def point6_run_apriori(min_support_count):
    min_support = min_support_count / point6_n
    itemsets, _ = apriori(
        point6_transactions,
        min_support=min_support,
        min_confidence=1.0,
    )

    support_lookup = {}
    for level in itemsets.values():
        for itemset, support_count in level.items():
            support_lookup[point6_normalize_itemset(itemset)] = support_count

    return support_lookup


def point6_run_fpgrowth(min_support_count):
    patterns = pyfpgrowth.find_frequent_patterns(point6_transactions, min_support_count)
    return {
        point6_normalize_itemset(itemset): support_count
        for itemset, support_count in patterns.items()
    }


point6_itemsets_rows = []
point6_rules_rows = []
point6_timing_rows = []
point6_itemset_summary_rows = []
point6_rule_summary_rows = []
point6_itemset_consistency_rows = []
point6_rule_consistency_rows = []

for min_support_count in point6_support_counts:
    apriori_lookup, apriori_itemset_time = point6_average_runtime(
        lambda count=min_support_count: point6_run_apriori(count)
    )
    fpgrowth_lookup, fpgrowth_itemset_time = point6_average_runtime(
        lambda count=min_support_count: point6_run_fpgrowth(count)
    )

    point6_timing_rows.extend(
        [
            {
                "algoritmo": "Apriori",
                "min_support_count": min_support_count,
                "fase": "itemsets frecuentes",
                "elapsed_seconds": apriori_itemset_time,
            },
            {
                "algoritmo": "FP-Growth",
                "min_support_count": min_support_count,
                "fase": "itemsets frecuentes",
                "elapsed_seconds": fpgrowth_itemset_time,
            },
        ]
    )

    point6_itemsets_rows.extend(
        point6_itemset_rows_from_lookup(apriori_lookup, "Apriori", min_support_count)
    )
    point6_itemsets_rows.extend(
        point6_itemset_rows_from_lookup(fpgrowth_lookup, "FP-Growth", min_support_count)
    )

    point6_itemset_consistency_rows.append(
        {
            "min_support_count": min_support_count,
            "mismos_itemsets": apriori_lookup == fpgrowth_lookup,
        }
    )

    point6_rules_by_algorithm = {}

    for algorithm, support_lookup in [("Apriori", apriori_lookup), ("FP-Growth", fpgrowth_lookup)]:
        size_counts = {}
        for itemset in support_lookup:
            size_counts[len(itemset)] = size_counts.get(len(itemset), 0) + 1

        for size, total_itemsets in sorted(size_counts.items()):
            point6_itemset_summary_rows.append(
                {
                    "algoritmo": algorithm,
                    "min_support_count": min_support_count,
                    "size": size,
                    "num_itemsets": total_itemsets,
                }
            )

        generated_rule_sets, rule_time = point6_average_runtime(
            lambda lookup=support_lookup, name=algorithm, count=min_support_count: [
                point6_generate_rules_from_lookup(lookup, name, count, min_confidence)
                for min_confidence in point6_confidences
            ]
        )

        point6_timing_rows.append(
            {
                "algoritmo": algorithm,
                "min_support_count": min_support_count,
                "fase": "reglas para varias confianzas",
                "elapsed_seconds": rule_time,
            }
        )

        point6_rules_by_algorithm[algorithm] = {}

        for min_confidence, rule_rows in zip(point6_confidences, generated_rule_sets):
            point6_rules_rows.extend(rule_rows)
            point6_rules_by_algorithm[algorithm][min_confidence] = rule_rows
            point6_rule_summary_rows.append(
                {
                    "algoritmo": algorithm,
                    "min_support_count": min_support_count,
                    "min_confidence": min_confidence,
                    "num_rules": len(rule_rows),
                }
            )

    for min_confidence in point6_confidences:
        apriori_rules = point6_rules_by_algorithm["Apriori"][min_confidence]
        fpgrowth_rules = point6_rules_by_algorithm["FP-Growth"][min_confidence]

        apriori_signature = {
            (
                row["antecedente"],
                row["consecuente"],
                row["support_count_union"],
                row["confidence"],
            )
            for row in apriori_rules
        }

        fpgrowth_signature = {
            (
                row["antecedente"],
                row["consecuente"],
                row["support_count_union"],
                row["confidence"],
            )
            for row in fpgrowth_rules
        }

        point6_rule_consistency_rows.append(
            {
                "min_support_count": min_support_count,
                "min_confidence": min_confidence,
                "mismas_reglas": apriori_signature == fpgrowth_signature,
            }
        )

point6_itemsets_df = pd.DataFrame(point6_itemsets_rows).sort_values(
    ["min_support_count", "algoritmo", "size", "itemset"]
).reset_index(drop=True)

point6_itemset_summary_df = pd.DataFrame(point6_itemset_summary_rows).sort_values(
    ["min_support_count", "algoritmo", "size"]
).reset_index(drop=True)

point6_rules_df = pd.DataFrame(point6_rules_rows).sort_values(
    [
        "min_support_count",
        "min_confidence",
        "algoritmo",
        "antecedente_size",
        "antecedente",
        "consecuente",
        "confidence",
    ],
    ascending=[True, True, True, True, True, True, False],
).reset_index(drop=True)

point6_rule_summary_df = pd.DataFrame(point6_rule_summary_rows).sort_values(
    ["min_support_count", "algoritmo", "min_confidence"]
).reset_index(drop=True)

point6_itemset_consistency_df = pd.DataFrame(point6_itemset_consistency_rows)
point6_rule_consistency_df = pd.DataFrame(point6_rule_consistency_rows).sort_values(
    ["min_support_count", "min_confidence"]
).reset_index(drop=True)

point6_timing_df = pd.DataFrame(point6_timing_rows).sort_values(
    ["min_support_count", "fase", "algoritmo"]
).reset_index(drop=True)
point6_timing_df["elapsed_seconds"] = point6_timing_df["elapsed_seconds"].round(6)

for min_support_count in point6_support_counts:
    print(f"Itemsets frecuentes con soporte mínimo = {min_support_count}")
    for algorithm in ["Apriori", "FP-Growth"]:
        print(f"{algorithm}: itemsets frecuentes")
        display(
            point6_itemsets_df.loc[
                (point6_itemsets_df["min_support_count"] == min_support_count)
                & (point6_itemsets_df["algoritmo"] == algorithm),
                ["itemset", "size", "support_count", "support"],
            ]
        )

    print(f"Resumen de itemsets por tamaño para soporte mínimo = {min_support_count}")
    for algorithm in ["Apriori", "FP-Growth"]:
        print(f"{algorithm}: resumen por tamaño")
        display(
            point6_itemset_summary_df.loc[
                (point6_itemset_summary_df["min_support_count"] == min_support_count)
                & (point6_itemset_summary_df["algoritmo"] == algorithm),
                ["size", "num_itemsets"],
            ]
        )

    print(f"Número de reglas por nivel de confianza para soporte mínimo = {min_support_count}")
    display(
        point6_rule_summary_df.loc[
            point6_rule_summary_df["min_support_count"] == min_support_count
        ]
    )

    print(f"Comparación de reglas entre algoritmos para soporte mínimo = {min_support_count}")
    display(
        point6_rule_consistency_df.loc[
            point6_rule_consistency_df["min_support_count"] == min_support_count
        ]
    )

    for min_confidence in point6_confidences:
        print(
            f"Reglas ordenadas con soporte mínimo = {min_support_count} y confianza mínima = {min_confidence}"
        )
        display(
            point6_rules_df.loc[
                (point6_rules_df["min_support_count"] == min_support_count)
                & (point6_rules_df["min_confidence"] == min_confidence),
                [
                    "algoritmo",
                    "antecedente",
                    "consecuente",
                    "support_count_union",
                    "support_union",
                    "confidence",
                ],
            ]
        )

print("Comparación de itemsets entre algoritmos")
display(point6_itemset_consistency_df)

print("Comparación de tiempos promedio por ejecución")
display(point6_timing_df)


Itemsets frecuentes con soporte mínimo = 2
Apriori: itemsets frecuentes


,itemset,size,support_count,support
0,{A},1,8,0.6667
1,{B},1,7,0.5833
2,{C},1,5,0.4167
3,{D},1,7,0.5833
4,{E},1,7,0.5833
5,{F},1,5,0.4167
6,"{A, B}",2,4,0.3333
7,"{A, C}",2,3,0.2500
8,"{A, D}",2,3,0.2500
9,"{A, E}",2,3,0.2500


FP-Growth: itemsets frecuentes


,itemset,size,support_count,support
29,{A},1,8,0.6667
30,{B},1,7,0.5833
31,"{A, B}",2,4,0.3333
32,"{A, C}",2,3,0.2500
33,"{A, D}",2,3,0.2500
34,"{A, E}",2,3,0.2500
35,"{A, F}",2,4,0.3333
36,"{B, C}",2,2,0.1667
37,"{B, D}",2,5,0.4167
38,"{B, E}",2,4,0.3333


Resumen de itemsets por tamaño para soporte mínimo = 2
Apriori: resumen por tamaño


...

,algoritmo,antecedente,consecuente,support_count_union,support_union,confidence
456,Apriori,{D},{E},6,0.5000,0.8571
457,Apriori,{E},{D},6,0.5000,0.8571
458,Apriori,{F},{A},4,0.3333,0.8000
459,Apriori,"{B, D}",{E},4,0.3333,0.8000
460,Apriori,"{B, E}",{D},4,0.3333,1.0000
461,Apriori,"{C, D}",{E},3,0.2500,1.0000
462,Apriori,"{C, E}",{D},3,0.2500,1.0000
463,FP-Growth,{D},{E},6,0.5000,0.8571
464,FP-Growth,{E},{D},6,0.5000,0.8571
465,FP-Growth,{F},{A},4,0.3333,0.8000


Comparación de itemsets entre algoritmos


,min_support_count,mismos_itemsets
0,2,False
1,3,False


Comparación de tiempos promedio por ejecución


,algoritmo,min_support_count,fase,elapsed_seconds
0,Apriori,2,itemsets frecuentes,0.000283
1,FP-Growth,2,itemsets frecuentes,0.000224
2,Apriori,2,reglas para varias confianzas,0.001562
3,FP-Growth,2,reglas para varias confianzas,0.001673
4,Apriori,3,itemsets frecuentes,0.000123
5,FP-Growth,3,itemsets frecuentes,0.000178
6,Apriori,3,reglas para varias confianzas,0.000610
7,FP-Growth,3,reglas para varias confianzas,0.000612


**Resultados con soporte mínimo igual a 2**
- Tanto Apriori como FP-Growth recuperan los mismos `29` itemsets frecuentes después de normalizar la salida.
- La distribución por tamaño es: `6` itemsets de tamaño `1`, `14` de tamaño `2`, `8` de tamaño `3` y `1` de tamaño `4`.
- El itemset más grande que sobrevive con este soporte es `{B, D, E, F}`, con soporte absoluto `2`.
- El número de reglas fuertes disminuye al subir la confianza: `64` con `0.4`, `46` con `0.5`, `32` con `0.6` y `16` con `0.8`.
- Entre las reglas más fuertes aparecen `{D} -> {E}`, `{E} -> {D}`, `{F} -> {A}`, `{B, E} -> {D}`, `{C, D} -> {E}` y `{C, E} -> {D}`.

**Resultados con soporte mínimo igual a 3**
- Con este umbral, ambos algoritmos coinciden en `19` itemsets frecuentes.
- La distribución por tamaño pasa a `6` itemsets de tamaño `1`, `11` de tamaño `2` y solo `2` de tamaño `3`.
- Desaparecen el itemset de tamaño `4` y varios triples que sí eran frecuentes con soporte `2`; solo permanecen `{B, D, E}` y `{C, D, E}`.
- El total de reglas fuertes vuelve a caer: `31` con `0.4`, `24` con `0.5`, `15` con `0.6` y `7` con `0.8`.
- A confianza `0.8`, las reglas más representativas son `{D} -> {E}`, `{E} -> {D}`, `{F} -> {A}`, `{B, D} -> {E}`, `{B, E} -> {D}`, `{C, D} -> {E}` y `{C, E} -> {D}`.

**Comparación entre Apriori y FP-Growth**
- La tabla `point6_itemset_consistency_df` muestra si ambos algoritmos devolvieron exactamente el mismo conjunto de itemsets frecuentes para cada soporte.
- La tabla `point6_rule_consistency_df` muestra si, después de normalizar la salida, ambos algoritmos producen el mismo conjunto de reglas para cada nivel de confianza.
- Para comparar reglas sin depender del formato de salida de cada librería, el notebook las reconstruye desde los itemsets frecuentes normalizados de cada algoritmo.
- La tabla `point6_timing_df` resume el tiempo promedio por ejecución. En una base de `12` transacciones la diferencia suele ser pequeña, pero sigue siendo útil para ver cuál algoritmo fue más eficiente.
- Al subir el soporte de `2` a `3` se reducen los itemsets frecuentes, se podan reglas y el análisis se vuelve más compacto en ambos algoritmos.


## 7. Tablas de contingencia y ranking de reglas

Usando el conjunto de datos del punto 5:

- **(a)** realizar la tabla de contingencia para las siguientes reglas: `b -> c`, `a -> d`, `b -> d`, `e -> c`, `c -> a`.
- **(b)** usar las tablas de contingencia del punto anterior para computar y realizar un ranking de las reglas usando:
  - soporte
  - confianza
  - lift


In [174]:
assert "point5_transactions" in globals(), "Ejecuta primero la celda de verificación del punto 5."

point7_rules = [("b", "c"), ("a", "d"), ("b", "d"), ("e", "c"), ("c", "a")]


def point7_rule_label(antecedent, consequent):
    return f"{antecedent} -> {consequent}"


def point7_contingency_counts(antecedent, consequent):
    n11 = sum((antecedent in transaction) and (consequent in transaction) for transaction in point5_transactions)
    n10 = sum((antecedent in transaction) and (consequent not in transaction) for transaction in point5_transactions)
    n01 = sum((antecedent not in transaction) and (consequent in transaction) for transaction in point5_transactions)
    n00 = sum((antecedent not in transaction) and (consequent not in transaction) for transaction in point5_transactions)
    return n11, n10, n01, n00


def point7_rule_metrics(n11, n10, n01, n00):
    support = round(n11 / point5_n, 4)
    confidence = round(n11 / (n11 + n10), 4) if (n11 + n10) else 0.0
    consequent_support = (n11 + n01) / point5_n
    lift = round(confidence / consequent_support, 4) if consequent_support else 0.0
    return support, confidence, lift


point7_contingency_rows = []
point7_metrics_rows = []
point7_matrix_tables = {}

for antecedent, consequent in point7_rules:
    rule = point7_rule_label(antecedent, consequent)
    n11, n10, n01, n00 = point7_contingency_counts(antecedent, consequent)
    support, confidence, lift = point7_rule_metrics(n11, n10, n01, n00)

    point7_contingency_rows.append(
        {
            "regla": rule,
            "X": antecedent,
            "Y": consequent,
            "n11": n11,
            "n10": n10,
            "n01": n01,
            "n00": n00,
        }
    )

    point7_metrics_rows.append(
        {
            "regla": rule,
            "support": support,
            "confidence": confidence,
            "lift": lift,
        }
    )

    point7_matrix_tables[rule] = pd.DataFrame(
        [[n11, n10], [n01, n00]],
        index=[f"{antecedent}=Sí", f"{antecedent}=No"],
        columns=[f"{consequent}=Sí", f"{consequent}=No"],
    )

point7_contingency_df = pd.DataFrame(point7_contingency_rows)
point7_metrics_df = pd.DataFrame(point7_metrics_rows)

point7_support_ranking_df = point7_metrics_df.sort_values(
    ["support", "confidence", "lift", "regla"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
point7_support_ranking_df.insert(
    0,
    "puesto",
    point7_support_ranking_df["support"].rank(method="dense", ascending=False).astype(int),
)

point7_confidence_ranking_df = point7_metrics_df.sort_values(
    ["confidence", "support", "lift", "regla"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
point7_confidence_ranking_df.insert(
    0,
    "puesto",
    point7_confidence_ranking_df["confidence"].rank(method="dense", ascending=False).astype(int),
)

point7_lift_ranking_df = point7_metrics_df.sort_values(
    ["lift", "confidence", "support", "regla"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
point7_lift_ranking_df.insert(
    0,
    "puesto",
    point7_lift_ranking_df["lift"].rank(method="dense", ascending=False).astype(int),
)

for rule, contingency_table in point7_matrix_tables.items():
    print(f"Tabla de contingencia para {rule}")
    display(contingency_table)

print("Resumen de tablas de contingencia")
display(point7_contingency_df)

print("Métricas de las reglas")
display(point7_metrics_df)

print("Ranking por soporte")
display(point7_support_ranking_df)

print("Ranking por confianza")
display(point7_confidence_ranking_df)

print("Ranking por lift")
display(point7_lift_ranking_df)


Tabla de contingencia para b -> c


,c=Sí,c=No
b=Sí,3,4
b=No,2,1


Tabla de contingencia para a -> d


,d=Sí,d=No
a=Sí,4,1
a=No,5,0


Tabla de contingencia para b -> d


...

,puesto,regla,support,confidence,lift
0,1,b -> d,0.6,0.8571,0.9523
1,2,a -> d,0.4,0.8000,0.8889
2,3,b -> c,0.3,0.4286,0.8572
3,4,c -> a,0.2,0.4000,0.8000
4,4,e -> c,0.2,0.3333,0.6666


Ranking por confianza


,puesto,regla,support,confidence,lift
0,1,b -> d,0.6,0.8571,0.9523
1,2,a -> d,0.4,0.8000,0.8889
2,3,b -> c,0.3,0.4286,0.8572
3,4,c -> a,0.2,0.4000,0.8000
4,5,e -> c,0.2,0.3333,0.6666


Ranking por lift


,puesto,regla,support,confidence,lift
0,1,b -> d,0.6,0.8571,0.9523
1,2,a -> d,0.4,0.8000,0.8889
2,3,b -> c,0.3,0.4286,0.8572
3,4,c -> a,0.2,0.4000,0.8000
4,5,e -> c,0.2,0.3333,0.6666


**Interpretación**
- La regla más sólida entre las cinco es `b -> d`, porque lidera las tres métricas comparadas.
- Sin embargo, ninguna de las reglas tiene `lift > 1`, por lo que ninguna muestra una asociación positiva fuerte; todas están por debajo de lo que se esperaría si la coocurrencia aportara evidencia favorable clara.
- La regla más débil es `e -> c`, ya que queda última en confianza y en lift.


## 8. Market Basket: reglas de asociación y filtro de atributos

Usando el mismo conjunto de datos del punto 1:

- extraer reglas de asociación utilizando diferentes valores mínimos de confianza;
- reportar los itemsets frecuentes con sus respectivos valores de soporte;
- analizar el problema que aparece y discutir las diferencias con el esquema que no posee un filtro de atributos.

En este punto se comparan dos esquemas:
- **Con filtro de atributos**: solo se conservan los productos con valor `True`.
- **Sin filtro de atributos**: se conserva `id` y cada atributo se transforma en pares `atributo=valor`.


In [175]:
from collections import Counter
from itertools import combinations

assert "basket" in globals(), "Ejecuta primero la celda de carga del punto 1."

point8_base_df = basket.copy()
point8_item_columns = [column for column in point8_base_df.columns if column != "id"]
point8_min_support_count = 2
point8_confidences = [0.4, 0.6, 0.8]


def point8_itemset_label(itemset):
    return "{" + ", ".join(itemset) + "}"


def point8_build_transactions(dataframe, filter_attributes):
    transactions = []
    for row in dataframe.to_dict(orient="records"):
        if filter_attributes:
            transaction = [column for column in point8_item_columns if bool(row[column])]
        else:
            transaction = [f"id={int(row['id'])}"]
            transaction.extend(
                [f"{column}={'true' if bool(row[column]) else 'false'}" for column in point8_item_columns]
            )

        transactions.append(tuple(sorted(transaction)))

    return transactions


def point8_support_count(itemset, transactions):
    itemset = set(itemset)
    return sum(itemset.issubset(set(transaction)) for transaction in transactions)


def point8_frequent_itemsets(transactions, min_support_count):
    counts = Counter()
    for transaction in transactions:
        unique_items = tuple(sorted(set(transaction)))
        for size in range(1, len(unique_items) + 1):
            for itemset in combinations(unique_items, size):
                counts[itemset] += 1

    frequent_lookup = {
        itemset: support_count
        for itemset, support_count in counts.items()
        if support_count >= min_support_count
    }

    return dict(sorted(frequent_lookup.items(), key=lambda pair: (len(pair[0]), pair[0])))


def point8_generate_rules(frequent_lookup, min_confidence, total_transactions):
    rows = []
    for itemset, support_count_union in frequent_lookup.items():
        if len(itemset) < 2:
            continue

        itemset_set = set(itemset)
        for lhs_size in range(1, len(itemset)):
            for lhs in combinations(itemset, lhs_size):
                lhs = tuple(sorted(lhs))
                rhs = tuple(sorted(itemset_set.difference(lhs)))
                confidence = round(support_count_union / frequent_lookup[lhs], 4)

                if confidence >= min_confidence:
                    rows.append(
                        {
                            "antecedente": point8_itemset_label(lhs),
                            "consecuente": point8_itemset_label(rhs),
                            "antecedente_size": len(lhs),
                            "support_count_union": support_count_union,
                            "support_union": round(support_count_union / total_transactions, 4),
                            "confidence": confidence,
                        }
                    )

    return sorted(
        rows,
        key=lambda row: (
            -row["confidence"],
            -row["support_union"],
            row["antecedente_size"],
            row["antecedente"],
            row["consecuente"],
        ),
    )


point8_transactions_by_scheme = {
    "Con filtro de atributos": point8_build_transactions(point8_base_df, filter_attributes=True),
    "Sin filtro de atributos": point8_build_transactions(point8_base_df, filter_attributes=False),
}

point8_itemsets_rows = []
point8_itemset_summary_rows = []
point8_rules_rows = []
point8_rule_summary_rows = []
point8_identifier_support_rows = []

for scheme, transactions in point8_transactions_by_scheme.items():
    frequent_lookup = point8_frequent_itemsets(transactions, point8_min_support_count)

    for itemset, support_count in frequent_lookup.items():
        point8_itemsets_rows.append(
            {
                "esquema": scheme,
                "itemset": point8_itemset_label(itemset),
                "size": len(itemset),
                "support_count": support_count,
                "support": round(support_count / len(transactions), 4),
            }
        )

    size_counts = Counter(len(itemset) for itemset in frequent_lookup)
    for size, total_itemsets in sorted(size_counts.items()):
        point8_itemset_summary_rows.append(
            {
                "esquema": scheme,
                "size": size,
                "num_itemsets": total_itemsets,
            }
        )

    for min_confidence in point8_confidences:
        rules = point8_generate_rules(frequent_lookup, min_confidence, len(transactions))
        point8_rule_summary_rows.append(
            {
                "esquema": scheme,
                "min_confidence": min_confidence,
                "num_rules": len(rules),
            }
        )

        for row in rules:
            point8_rules_rows.append(
                {
                    "esquema": scheme,
                    "min_confidence": min_confidence,
                    **row,
                }
            )

for identifier in point8_base_df["id"].tolist():
    item = (f"id={identifier}",)
    support_count = point8_support_count(item, point8_transactions_by_scheme["Sin filtro de atributos"])
    point8_identifier_support_rows.append(
        {
            "item": point8_itemset_label(item),
            "support_count": support_count,
            "support": round(support_count / len(point8_transactions_by_scheme["Sin filtro de atributos"]), 4),
        }
    )

point8_itemsets_df = pd.DataFrame(point8_itemsets_rows).sort_values(
    ["esquema", "size", "support_count", "itemset"],
    ascending=[True, True, False, True],
).reset_index(drop=True)

point8_itemset_summary_df = pd.DataFrame(point8_itemset_summary_rows).sort_values(
    ["esquema", "size"]
).reset_index(drop=True)

point8_rules_df = pd.DataFrame(point8_rules_rows).sort_values(
    ["esquema", "min_confidence", "confidence", "support_union", "antecedente", "consecuente"],
    ascending=[True, True, False, False, True, True],
).reset_index(drop=True)

point8_rule_summary_df = pd.DataFrame(point8_rule_summary_rows).sort_values(
    ["esquema", "min_confidence"]
).reset_index(drop=True)

point8_identifier_support_df = pd.DataFrame(point8_identifier_support_rows)

for scheme in point8_transactions_by_scheme:
    print(f"Esquema: {scheme}")
    print("Resumen de itemsets frecuentes por tamaño")
    display(
        point8_itemset_summary_df.loc[
            point8_itemset_summary_df["esquema"] == scheme,
            ["size", "num_itemsets"],
        ]
    )

    print("Itemsets frecuentes con soporte")
    display(
        point8_itemsets_df.loc[
            point8_itemsets_df["esquema"] == scheme,
            ["itemset", "size", "support_count", "support"],
        ]
    )

    print("Número de reglas por confianza mínima")
    display(
        point8_rule_summary_df.loc[
            point8_rule_summary_df["esquema"] == scheme
        ]
    )


print("Soporte de los identificadores en el esquema sin filtro")
display(point8_identifier_support_df)


Esquema: Con filtro de atributos
Resumen de itemsets frecuentes por tamaño


,size,num_itemsets
0,1,6
1,2,9
2,3,4
3,4,1


Itemsets frecuentes con soporte


,itemset,size,support_count,support
0,{panales},1,7,0.7
1,{leche},1,5,0.5
2,{mantequilla},1,5,0.5
3,{pan},1,5,0.5
4,{cerveza},1,4,0.4
5,{galletas},1,4,0.4
6,"{mantequilla, pan}",2,5,0.5
7,"{leche, panales}",2,4,0.4
8,"{cerveza, panales}",2,3,0.3
9,"{leche, mantequilla}",2,3,0.3


Número de reglas por confianza mínima


...

,itemset,size,support_count,support
20,{panales=true},1,7,0.7
21,{cerveza=false},1,6,0.6
22,{galletas=false},1,6,0.6
23,{leche=false},1,5,0.5
24,{leche=true},1,5,0.5
...,...,...,...,...
169,"{cerveza=true, galletas=false, mantequilla=fal...",5,2,0.2
170,"{cerveza=true, galletas=true, leche=false, man...",5,2,0.2
171,"{cerveza=true, leche=false, mantequilla=false,...",5,2,0.2
172,"{galletas=false, leche=true, mantequilla=true,...",5,2,0.2


Número de reglas por confianza mínima


,esquema,min_confidence,num_rules
3,Sin filtro de atributos,0.4,1126
4,Sin filtro de atributos,0.6,743
5,Sin filtro de atributos,0.8,311


Soporte de los identificadores en el esquema sin filtro


,item,support_count,support
0,{id=1},1,0.1
1,{id=2},1,0.1
2,{id=3},1,0.1
3,{id=4},1,0.1
4,{id=5},1,0.1
5,{id=6},1,0.1
6,{id=7},1,0.1
7,{id=8},1,0.1
8,{id=9},1,0.1
9,{id=10},1,0.1


**Esquema con filtro de atributos**
- Este esquema reutiliza exactamente las compras reales del punto 1: solo se conservan los productos con valor `True`.
- Con soporte mínimo `2`, aparecen `20` itemsets frecuentes: `6` de tamaño `1`, `9` de tamaño `2`, `4` de tamaño `3` y `1` de tamaño `4`.
- Los itemsets más representativos son `{panales}` con soporte `0.7`, `{mantequilla, pan}` con soporte `0.5` y `{leche, panales}` con soporte `0.4`.
- También se observa que los mayores soportes quedan concentrados en itemsets individuales, y que al aumentar el número de ítems el soporte tiende a disminuir porque cada combinación más específica aparece en menos transacciones.
- El número de reglas es razonable y disminuye al subir la confianza: `52` con `0.4`, `33` con `0.6` y `9` con `0.8`.
- Entre las reglas más útiles aparecen `mantequilla -> pan`, `pan -> mantequilla` y `leche -> panales`, que sí se interpretan como patrones de compra.

**Esquema sin filtro de atributos**
- En este esquema se conserva `id` y cada atributo se convierte en pares `atributo=valor`, por ejemplo `leche=true`, `leche=false` o `pan=false`.
- Con el mismo soporte mínimo aparecen `154` itemsets frecuentes: `12` de tamaño `1`, `45` de tamaño `2`, `55` de tamaño `3`, `32` de tamaño `4`, `9` de tamaño `5` y `1` de tamaño `6`.
- Los itemsets y reglas quedan dominados por valores `false`, por ejemplo `cerveza=false`, `galletas=false`, `mantequilla=false -> pan=false` o `mantequilla=true -> cerveza=false`.
- Además, los identificadores `id=k` tienen soporte `1`, así que no aportan itemsets frecuentes con este umbral, pero sí amplían el espacio de atributos sin aportar información útil.
- El número de reglas se dispara: `1126` con `0.4`, `743` con `0.6` y `311` con `0.8`.

**Problema y comparación final**
- El problema que se presenta es semántico: sin filtro, el operador trata ausencias de compra (`false`) e identificadores como si fueran ítems del market basket.
- Eso produce soportes correctos desde el punto de vista aritmético, pero reglas poco útiles o engañosas para analizar compras reales.
- Además, al ordenar por soporte, los itemsets de tamaño `1` suelen dominar la salida y los itemsets más grandes van perdiendo soporte, así que el soporte por sí solo no basta para decidir qué asociaciones son las más interesantes.
- El esquema con filtro de atributos es el adecuado para market basket porque conserva solo presencia de productos; el esquema sin filtro genera muchos más itemsets y reglas, pero la mayoría no describe comportamiento de compra sino combinaciones de valores del registro.


## 9. German Credit: discretización y reglas de asociación

Importe el conjunto de datos `credit-german.csv` del repositorio UCI. Discretice los atributos numéricos en máximo `5` bins de igual tamaño. Aplique el algoritmo de reglas de asociación a este conjunto, interprete las reglas producidas, varíe los valores de soporte y confianza y escoja las reglas que resulten más interesantes para el problema de crédito.

En este punto se usa la versión clásica **Statlog (German Credit Data)** de UCI, se incluyen los atributos discretizados dentro de las transacciones y también se incluye la clase objetivo para poder encontrar reglas asociadas a `good` y `bad` crédito.


In [176]:
point9_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"

point9_column_names = [
    "checking_status",
    "duration",
    "credit_history",
    "purpose",
    "credit_amount",
    "savings_status",
    "employment",
    "installment_commitment",
    "personal_status",
    "other_parties",
    "residence_since",
    "property_magnitude",
    "age",
    "other_payment_plans",
    "housing",
    "existing_credits",
    "job",
    "num_dependents",
    "own_telephone",
    "foreign_worker",
    "credit_class",
]

point9_numeric_columns = [
    "duration",
    "credit_amount",
    "installment_commitment",
    "residence_since",
    "age",
    "existing_credits",
    "num_dependents",
]

point9_support_levels = [0.05, 0.10, 0.15]
point9_confidence_levels = [0.60, 0.70, 0.80]
point9_balanced_support = 0.10
point9_balanced_confidence = 0.70
point9_n_rules_preview = 15

point9_value_labels = {
    "checking_status": {
        "A11": "lt_0_dm",
        "A12": "between_0_and_200_dm",
        "A13": "gte_200_dm",
        "A14": "no_checking",
    },
    "credit_history": {
        "A30": "no_credits_taken_all_paid",
        "A31": "all_credits_paid_back_duly",
        "A32": "existing_credits_paid_duly",
        "A33": "delay_in_paying_off",
        "A34": "critical_account_other_credits",
    },
    "purpose": {
        "A40": "car_new",
        "A41": "car_used",
        "A42": "furniture_equipment",
        "A43": "radio_tv",
        "A44": "domestic_appliances",
        "A45": "repairs",
        "A46": "education",
        "A48": "retraining",
        "A49": "business",
        "A410": "others",
    },
    "savings_status": {
        "A61": "lt_100_dm",
        "A62": "between_100_and_500_dm",
        "A63": "between_500_and_1000_dm",
        "A64": "gte_1000_dm",
        "A65": "unknown_no_savings",
    },
    "employment": {
        "A71": "unemployed",
        "A72": "lt_1_year",
        "A73": "between_1_and_4_years",
        "A74": "between_4_and_7_years",
        "A75": "gte_7_years",
    },
    "personal_status": {
        "A91": "male_divorced_separated",
        "A92": "female_divorced_separated_married",
        "A93": "male_single",
        "A94": "male_married_widowed",
        "A95": "female_single",
    },
    "other_parties": {
        "A101": "none",
        "A102": "co_applicant",
        "A103": "guarantor",
    },
    "property_magnitude": {
        "A121": "real_estate",
        "A122": "life_insurance",
        "A123": "car_or_other",
        "A124": "unknown_no_property",
    },
    "other_payment_plans": {
        "A141": "bank",
        "A142": "stores",
        "A143": "none",
    },
    "housing": {
        "A151": "rent",
        "A152": "own",
        "A153": "for_free",
    },
    "job": {
        "A171": "unemployed_non_resident",
        "A172": "unskilled_resident",
        "A173": "skilled_employee",
        "A174": "management_self_employed",
    },
    "own_telephone": {
        "A191": "none",
        "A192": "yes_registered",
    },
    "foreign_worker": {
        "A201": "yes",
        "A202": "no",
    },
    "credit_class": {
        1: "good",
        2: "bad",
        "1": "good",
        "2": "bad",
    },
}

point9_raw_df = pd.read_csv(
    point9_url,
    sep=r"\s+",
    header=None,
    names=point9_column_names,
    engine="python",
)

point9_df = point9_raw_df.copy()
point9_df["credit_class"] = point9_df["credit_class"].map(point9_value_labels["credit_class"])

for column, mapping in point9_value_labels.items():
    if column == "credit_class":
        continue
    if column in point9_df.columns:
        point9_df[column] = point9_df[column].replace(mapping)

for column in point9_numeric_columns:
    point9_df[f"{column}_bin"] = pd.cut(
        point9_df[column],
        bins=5,
        duplicates="drop",
        include_lowest=True,
    ).astype(str)

point9_feature_columns = []
for column in point9_column_names:
    if column == "credit_class":
        continue
    if column in point9_numeric_columns:
        point9_feature_columns.append(f"{column}_bin")
    else:
        point9_feature_columns.append(column)

point9_transaction_columns = point9_feature_columns + ["credit_class"]
point9_transactions = [
    tuple(
        sorted(
            [f"{column}={row[column]}" for column in point9_transaction_columns]
        )
    )
    for _, row in point9_df[point9_transaction_columns].iterrows()
]
point9_n = len(point9_transactions)


def point9_itemset_label(itemset):
    return "{" + ", ".join(itemset) + "}"


def point9_rule_label(lhs, rhs):
    return f"{point9_itemset_label(lhs)} -> {point9_itemset_label(rhs)}"


def point9_extract_itemsets(itemsets_by_size, min_support):
    rows = []
    for size, level in itemsets_by_size.items():
        for itemset, support_count in level.items():
            normalized = tuple(sorted(itemset))
            rows.append(
                {
                    "min_support": min_support,
                    "itemset": point9_itemset_label(normalized),
                    "size": len(normalized),
                    "support_count": support_count,
                    "support": round(support_count / point9_n, 4),
                }
            )
    return rows


def point9_extract_rules(rules, min_support, min_confidence):
    rows = []
    for rule in rules:
        lhs = tuple(sorted(getattr(rule, "lhs", tuple())))
        rhs = tuple(sorted(getattr(rule, "rhs", tuple())))
        support = round(float(getattr(rule, "support", 0.0)), 4)
        confidence = round(float(getattr(rule, "confidence", 0.0)), 4)
        lift = round(float(getattr(rule, "lift", 0.0)), 4)
        support_count_union = int(getattr(rule, "count_full", round(support * point9_n)))
        rows.append(
            {
                "min_support": min_support,
                "min_confidence": min_confidence,
                "antecedente": point9_itemset_label(lhs),
                "consecuente": point9_itemset_label(rhs),
                "rule": point9_rule_label(lhs, rhs),
                "antecedente_size": len(lhs),
                "consecuente_size": len(rhs),
                "support_count_union": support_count_union,
                "support_union": support,
                "confidence": confidence,
                "lift": lift,
                "target_in_rhs": len(rhs) == 1 and rhs[0].startswith("credit_class="),
                "target_value": rhs[0] if len(rhs) == 1 and rhs[0].startswith("credit_class=") else "",
                "target_in_lhs": any(item.startswith("credit_class=") for item in lhs),
            }
        )
    return rows


point9_itemset_rows = []
point9_itemset_summary_rows = []
point9_rule_rows = []
point9_rule_summary_rows = []

for min_support in point9_support_levels:
    itemsets, _ = apriori(
        point9_transactions,
        min_support=min_support,
        min_confidence=min(point9_confidence_levels),
    )

    point9_itemset_rows.extend(point9_extract_itemsets(itemsets, min_support))

    for size, level in itemsets.items():
        point9_itemset_summary_rows.append(
            {
                "min_support": min_support,
                "size": size,
                "num_itemsets": len(level),
            }
        )

    for min_confidence in point9_confidence_levels:
        _, rules = apriori(
            point9_transactions,
            min_support=min_support,
            min_confidence=min_confidence,
        )
        extracted_rules = point9_extract_rules(rules, min_support, min_confidence)
        point9_rule_rows.extend(extracted_rules)
        point9_rule_summary_rows.append(
            {
                "min_support": min_support,
                "min_confidence": min_confidence,
                "num_rules": len(extracted_rules),
                "num_rules_to_target": sum(
                    row["target_in_rhs"] and not row["target_in_lhs"] for row in extracted_rules
                ),
            }
        )

point9_itemsets_df = pd.DataFrame(point9_itemset_rows).sort_values(
    ["min_support", "support_count", "size", "itemset"],
    ascending=[True, False, True, True],
).reset_index(drop=True)

point9_itemset_summary_df = pd.DataFrame(point9_itemset_summary_rows).sort_values(
    ["min_support", "size"]
).reset_index(drop=True)

point9_rules_df = pd.DataFrame(point9_rule_rows).sort_values(
    ["min_support", "min_confidence", "confidence", "lift", "support_union", "rule"],
    ascending=[True, True, False, False, False, True],
).reset_index(drop=True)

point9_rule_summary_df = pd.DataFrame(point9_rule_summary_rows).sort_values(
    ["min_support", "min_confidence"]
).reset_index(drop=True)

point9_target_rules_df = point9_rules_df.loc[
    point9_rules_df["target_in_rhs"] & ~point9_rules_df["target_in_lhs"]
].copy()

point9_balanced_rules_df = point9_target_rules_df.loc[
    (point9_target_rules_df["min_support"] == point9_balanced_support)
    & (point9_target_rules_df["min_confidence"] == point9_balanced_confidence)
].head(point9_n_rules_preview)

print("Dimensiones del dataset German Credit")
print(point9_raw_df.shape)

print("Vista previa de atributos discretizados")
display(point9_df[point9_transaction_columns].head())

print("Resumen de itemsets frecuentes por soporte mínimo")
display(point9_itemset_summary_df)

for min_support in point9_support_levels:
    print(f"Itemsets frecuentes más representativos con soporte mínimo = {min_support}")
    display(
        point9_itemsets_df.loc[
            point9_itemsets_df["min_support"] == min_support,
            ["itemset", "size", "support_count", "support"],
        ].head(20)
    )

print("Número de reglas por combinación de soporte y confianza")
display(point9_rule_summary_df)

print("Reglas con la clase objetivo como consecuente en el esquema balanceado")
display(
    point9_balanced_rules_df[
        [
            "rule",
            "support_union",
            "confidence",
            "lift",
            "target_value",
        ]
    ]
)


Dimensiones del dataset German Credit
(1000, 21)
Vista previa de atributos discretizados


,checking_status,duration_bin,credit_history,purpose,credit_amount_bin,savings_status,employment,installment_commitment_bin,personal_status,other_parties,...,property_magnitude,age_bin,other_payment_plans,housing,existing_credits_bin,job,num_dependents_bin,own_telephone,foreign_worker,credit_class
0,lt_0_dm,"(3.931, 17.6]",critical_account_other_credits,radio_tv,"(231.825, 3884.8]",unknown_no_savings,gte_7_years,"(3.4, 4.0]",male_single,none,...,real_estate,"(63.8, 75.0]",none,own,"(1.6, 2.2]",skilled_employee,"(0.998, 1.2]",yes_registered,yes,good
1,between_0_and_200_dm,"(44.8, 58.4]",existing_credits_paid_duly,radio_tv,"(3884.8, 7519.6]",lt_100_dm,between_1_and_4_years,"(1.6, 2.2]",female_divorced_separated_married,none,...,real_estate,"(18.942999999999998, 30.2]",none,own,"(0.996, 1.6]",skilled_employee,"(0.998, 1.2]",none,yes,bad
2,no_checking,"(3.931, 17.6]",critical_account_other_credits,education,"(231.825, 3884.8]",lt_100_dm,between_4_and_7_years,"(1.6, 2.2]",male_single,none,...,real_estate,"(41.4, 52.6]",none,own,"(0.996, 1.6]",unskilled_resident,"(1.8, 2.0]",none,yes,good
3,lt_0_dm,"(31.2, 44.8]",existing_credits_paid_duly,furniture_equipment,"(7519.6, 11154.4]",lt_100_dm,between_4_and_7_years,"(1.6, 2.2]",male_single,guarantor,...,life_insurance,"(41.4, 52.6]",none,for_free,"(0.996, 1.6]",skilled_employee,"(1.8, 2.0]",none,yes,good
4,lt_0_dm,"(17.6, 31.2]",delay_in_paying_off,car_new,"(3884.8, 7519.6]",lt_100_dm,between_1_and_4_years,"(2.8, 3.4]",male_single,none,...,unknown_no_property,"(52.6, 63.8]",none,for_free,"(1.6, 2.2]",skilled_employee,"(1.8, 2.0]",none,yes,bad


Resumen de itemsets frecuentes por soporte mínimo


,min_support,size,num_itemsets
0,0.05,1,68
1,0.05,2,1105
2,0.05,3,6879
3,0.05,4,21379
4,0.05,5,37840
5,0.05,6,40544
6,0.05,7,26465
7,0.05,8,10313
8,0.10,1,56
9,0.10,2,640


Itemsets frecuentes más representativos con soporte mínimo = 0.05


...

,itemset,size,support_count,support
168394,{foreign_worker=yes},1,963,0.963
168395,{other_parties=none},1,907,0.907
168396,"{foreign_worker=yes, other_parties=none}",2,880,0.880
168397,"{num_dependents_bin=(0.998, 1.2]}",1,845,0.845
168398,"{foreign_worker=yes, num_dependents_bin=(0.998...",2,819,0.819
168399,{other_payment_plans=none},1,814,0.814
168400,"{foreign_worker=yes, other_payment_plans=none}",2,782,0.782
168401,"{num_dependents_bin=(0.998, 1.2], other_partie...",2,767,0.767
168402,"{foreign_worker=yes, num_dependents_bin=(0.998...",3,749,0.749
168403,"{other_parties=none, other_payment_plans=none}",2,742,0.742


Número de reglas por combinación de soporte y confianza


,min_support,min_confidence,num_rules,num_rules_to_target
0,0.05,0.6,1496468,44372
1,0.05,0.7,935590,33702
2,0.05,0.8,535388,18207
3,0.10,0.6,213863,7046
4,0.10,0.7,135581,5123
5,0.10,0.8,77499,2025
6,0.15,0.6,58435,2046
7,0.15,0.7,37173,1475
8,0.15,0.8,21268,414


Reglas con la clase objetivo como consecuente en el esquema balanceado


,rule,support_union,confidence,lift,target_value
3182372,"{checking_status=no_checking, credit_history=c...",0.111,0.9911,1.4158,credit_class=good
3182489,"{checking_status=no_checking, credit_history=c...",0.108,0.9908,1.4155,credit_class=good
3182529,"{checking_status=no_checking, credit_history=c...",0.106,0.9907,1.4152,credit_class=good
3182665,"{checking_status=no_checking, credit_amount_bi...",0.104,0.9905,1.4150,credit_class=good
3182710,"{checking_status=no_checking, credit_history=c...",0.103,0.9904,1.4148,credit_class=good
3182830,"{checking_status=no_checking, credit_amount_bi...",0.101,0.9902,1.4146,credit_class=good
3182890,"{checking_status=no_checking, credit_amount_bi...",0.100,0.9901,1.4144,credit_class=good
3184682,"{checking_status=no_checking, housing=own, job...",0.101,0.9806,1.4008,credit_class=good
3184756,"{checking_status=no_checking, credit_amount_bi...",0.100,0.9804,1.4006,credit_class=good
3184757,"{checking_status=no_checking, credit_amount_bi...",0.100,0.9804,1.4006,credit_class=good


**Preparación del conjunto de datos**
- Para este punto se usa la versión clásica de **German Credit** del repositorio UCI.
- Los atributos numéricos se discretizan en máximo `5` bins de igual tamaño, lo que permite incluirlos dentro de las transacciones como categorías.
- La clase objetivo se mantiene como ítem (`credit_class=good` o `credit_class=bad`) para poder interpretar reglas útiles para el problema crediticio.

**Efecto de variar soporte y confianza**
- Al disminuir el soporte mínimo, aumenta el número de itemsets frecuentes y también crece el número de reglas candidatas.
- Al aumentar el soporte mínimo, sobreviven solo los patrones más comunes y el conjunto de reglas se reduce.
- Al aumentar la confianza mínima, las reglas se vuelven más estrictas y normalmente disminuye su cantidad.
- La tabla `point9_rule_summary_df` resume ese comportamiento para todas las combinaciones de soporte y confianza.

**Interpretación de las reglas**
- Las reglas más útiles para este problema son las que tienen la clase objetivo en el consecuente, porque permiten asociar perfiles de clientes con `good` o `bad` crédito.
- La tabla `point9_balanced_rules_df` muestra una selección balanceada de reglas con soporte `0.10` y confianza `0.70`, umbrales que suelen dejar reglas todavía frecuentes pero más interpretables que en soporte muy bajo.
- En este tipo de problema, las reglas más interesantes suelen involucrar variables como estado de cuenta, historial crediticio, monto, duración, ahorro, empleo y edad ya discretizada.

**Conclusión**
- La discretización permite incorporar variables numéricas al análisis de asociaciones, pero también aumenta el número de categorías posibles.
- Soportes bajos producen más reglas, aunque también más ruido; soportes altos producen menos reglas, pero generalmente más estables.
- Confianzas altas dejan menos reglas, pero suelen facilitar una interpretación más sólida.
- Para la selección final, conviene priorizar reglas con buen soporte, buena confianza y consecuente en la clase objetivo, porque son las que mejor ayudan a describir perfiles de riesgo de crédito.
